# Activation Checkpointing：手写分段重计算与 RNG 复现

**面试问题：激活重计算怎样换取显存，为什么随机状态和副作用会破坏梯度？**

## 回答主线

先明确业务目标和数据合同，再给出可比较的朴素基线；随后手写核心算法，展示中间状态、最终指标和失败路径。本 Notebook 的断言只出现在最后，用于保护关键不变量；学习重点是前面的输入、过程、对照与解释。

## 真实案例

训练一个长序列客服意图模型时，显存主要被 24 层隐藏状态占用。案例先用容量模型比较保存全部激活和每四层保存边界，再在六层可微链上手写完整 backward 与分段重计算 backward，逐参数对比梯度；最后故意更换 dropout mask，展示 RNG 不一致为何让恢复后的梯度悄悄漂移。

### 输入预览：24 层长序列训练配置

In [1]:
config = {"layers": 24, "batch": 4, "sequence": 2048, "hidden": 4096, "bytes_per_value": 2, "checkpoint_every": 4}  # 定义接近真实 LLM 微调的激活容量字段。
activation_mb = config["batch"] * config["sequence"] * config["hidden"] * config["bytes_per_value"] / 1024 ** 2  # 计算单层主隐藏状态的理论 MiB。
save_all_mb = activation_mb * (config["layers"] + 1)  # 估算保存输入和每层输出的基线激活容量。
boundary_count = config["layers"] // config["checkpoint_every"] + 1  # 计算分段重计算需要保留的边界数量。
checkpoint_mb = activation_mb * boundary_count  # 估算只保存分段边界的激活容量。
print("训练配置：", config)  # 展示容量计算依赖的全部输入字段。
print(f"单层隐藏状态={activation_mb:.1f} MiB，保存全部={save_all_mb:.1f} MiB，分段边界={checkpoint_mb:.1f} MiB")  # 直观看到 checkpointing 的显存收益。

训练配置： {'layers': 24, 'batch': 4, 'sequence': 2048, 'hidden': 4096, 'bytes_per_value': 2, 'checkpoint_every': 4}
单层隐藏状态=64.0 MiB，保存全部=1600.0 MiB，分段边界=448.0 MiB


## Baseline 基线：保存六层链的每个中间激活

In [2]:
import numpy as np  # 导入数组运算以手写前向和反向传播。

rng = np.random.default_rng(23)  # 固定随机源保证权重和 dropout mask 可复现。
x = rng.normal(size=5)  # 构造五维输入向量模拟一个压缩后的 token 表示。
weights = [rng.normal(loc=0.9, scale=0.08, size=5) for _ in range(6)]  # 构造六层逐元素缩放权重。
biases = [rng.normal(scale=0.05, size=5) for _ in range(6)]  # 构造六层偏置参数。
masks = [(rng.random(5) > 0.2).astype(float) / 0.8 for _ in range(6)]  # 预先生成 inverted-dropout mask 作为 RNG 状态。

def full_forward_backward(inputs, layer_weights, layer_biases, dropout_masks):  # 实现保存全部激活的教学基线。
    states = [inputs.copy()]  # 保存输入作为第零层激活。
    tanh_values = []  # 保存每层 tanh 输出供反向导数使用。
    for weight, bias, mask in zip(layer_weights, layer_biases, dropout_masks):  # 顺序执行六层前向计算。
        tanh_value = np.tanh(states[-1] * weight + bias)  # 计算当前层的非线性输出。
        tanh_values.append(tanh_value)  # 保存未乘 dropout 的值用于导数。
        states.append(tanh_value * mask)  # 应用固定 mask 并保存层输出。
    gradient = np.ones_like(states[-1]) / states[-1].size  # 以输出均值作为标量损失并初始化梯度。
    weight_gradients = [np.zeros_like(weight) for weight in layer_weights]  # 预分配各层权重梯度。
    for index in range(len(layer_weights) - 1, -1, -1):  # 从最后一层向输入方向手写反向传播。
        local = gradient * dropout_masks[index] * (1.0 - tanh_values[index] ** 2)  # 计算 dropout 与 tanh 的链式梯度。
        weight_gradients[index] = local * states[index]  # 计算逐元素权重梯度。
        gradient = local * layer_weights[index]  # 把梯度传给前一层激活。
    return states[-1], weight_gradients, len(states)  # 返回输出、全部梯度和保存激活数量。

full_output, full_gradients, full_saved = full_forward_backward(x, weights, biases, masks)  # 运行保存全部状态的基线实现。
print("基线输出：", np.round(full_output, 5))  # 展示六层链实际产生的输出向量。
print(f"基线保存激活数量={full_saved}，第一层梯度={np.round(full_gradients[0], 6)}")  # 展示内存状态与可核验梯度。

基线输出： [ 0.       0.04697 -0.      -0.89361  0.12103]
基线保存激活数量=7，第一层梯度=[ 0.        0.       -0.       -0.001166  0.      ]


### 核心实现：只保存分段边界并在 backward 重算

In [3]:
def checkpoint_forward(inputs, layer_weights, layer_biases, dropout_masks, segment=2):  # 实现只保存每个分段边界的前向过程。
    boundaries = {0: inputs.copy()}  # 保存第零层输入作为首个边界。
    state = inputs.copy()  # 初始化当前层状态。
    calls = 0  # 统计前向层调用次数用于计算重算成本。
    for index, (weight, bias, mask) in enumerate(zip(layer_weights, layer_biases, dropout_masks), start=1):  # 顺序计算每一层但不保留内部激活。
        state = np.tanh(state * weight + bias) * mask  # 执行和基线完全相同的确定性层变换。
        calls += 1  # 累加一次真实层调用。
        if index % segment == 0 or index == len(layer_weights):  # 只在分段末尾保存激活。
            boundaries[index] = state.copy()  # 保存重计算所需的最小边界状态。
    return state, boundaries, calls  # 返回最终输出、边界字典和前向调用数。

def checkpoint_backward(boundaries, layer_weights, layer_biases, dropout_masks, segment=2):  # 实现从边界重算局部激活的反向过程。
    gradient = np.ones_like(boundaries[len(layer_weights)]) / boundaries[len(layer_weights)].size  # 初始化输出均值损失的梯度。
    weight_gradients = [np.zeros_like(weight) for weight in layer_weights]  # 预分配每层权重梯度。
    recompute_calls = 0  # 统计 backward 阶段额外执行的层次数。
    for end in range(len(layer_weights), 0, -segment):  # 按反向顺序逐段处理网络。
        start = max(0, end - segment)  # 计算当前分段的起始层索引。
        local_states = [boundaries[start].copy()]  # 从已保存边界恢复当前段输入。
        local_tanh = []  # 保存仅当前段的 tanh 值供局部 backward 使用。
        for index in range(start, end):  # 重新执行当前段的前向层。
            tanh_value = np.tanh(local_states[-1] * layer_weights[index] + layer_biases[index])  # 重建该层非线性输出。
            local_tanh.append(tanh_value)  # 保存当前段内的临时导数状态。
            local_states.append(tanh_value * dropout_masks[index])  # 使用原 mask 重建该层输出。
            recompute_calls += 1  # 记录一次重计算开销。
        for index in range(end - 1, start - 1, -1):  # 在当前段内部执行反向传播。
            offset = index - start  # 把全局层索引映射到局部缓存索引。
            local = gradient * dropout_masks[index] * (1.0 - local_tanh[offset] ** 2)  # 计算当前层局部梯度。
            weight_gradients[index] = local * local_states[offset]  # 计算当前层权重梯度。
            gradient = local * layer_weights[index]  # 把梯度传到前一层或前一分段。
    return weight_gradients, recompute_calls  # 返回重计算梯度和额外层调用数。

checkpoint_output, boundaries, forward_calls = checkpoint_forward(x, weights, biases, masks)  # 运行只保存边界的前向过程。
checkpoint_gradients, recompute_calls = checkpoint_backward(boundaries, weights, biases, masks)  # 运行手写分段重计算 backward。
max_gradient_error = max(float(np.max(np.abs(left - right))) for left, right in zip(full_gradients, checkpoint_gradients))  # 汇总两条路径的最大梯度误差。
print("保存的边界层：", sorted(boundaries))  # 展示哪些激活真正驻留在内存中。
print(f"前向调用={forward_calls}，反向重算调用={recompute_calls}，最大梯度误差={max_gradient_error:.3e}")  # 展示显存与计算的核心交换。

保存的边界层： [0, 2, 4, 6]
前向调用=6，反向重算调用=6，最大梯度误差=0.000e+00


## 结果解读：显存下降并不等于计算免费

In [4]:
memory_reduction = 1.0 - checkpoint_mb / save_all_mb  # 计算理论激活显存下降比例。
compute_multiplier = (forward_calls + recompute_calls) / forward_calls  # 计算教学实现的层调用放大倍数。
comparison_rows = [  # 构造可以直接阅读的基线与重计算对照表。
    ("保存全部", full_saved, 6, 0, "精确基线"),  # 基线保存七个状态且 backward 不重算层。
    ("每两层 checkpoint", len(boundaries), forward_calls, recompute_calls, f"误差 {max_gradient_error:.1e}"),  # 分段方案只保存四个边界但重算六层。
]  # 完成对照表数据。
print("方案                 保存状态  前向层调用  重算层调用  梯度结果")  # 输出对照表表头。
for name, saved, forward, recompute, result in comparison_rows:  # 逐行渲染显存、计算和正确性指标。
    print(f"{name:<20} {saved:>4} {forward:>10} {recompute:>10}  {result}")  # 展示 checkpointing 的真实成本收益。
print(f"24 层容量模型估算显存下降={memory_reduction:.1%}；教学链层调用放大={compute_multiplier:.1f}x")  # 把小型正确性实验连接到实际容量规划。

方案                 保存状态  前向层调用  重算层调用  梯度结果
保存全部                    7          6          0  精确基线
每两层 checkpoint          4          6          6  误差 0.0e+00
24 层容量模型估算显存下降=72.0%；教学链层调用放大=2.0x


## 失败案例：重计算时没有恢复 Dropout RNG

In [5]:
wrong_masks = [(rng.random(5) > 0.2).astype(float) / 0.8 for _ in range(6)]  # 故意生成与原前向不同的 dropout mask。
wrong_gradients, wrong_calls = checkpoint_backward(boundaries, weights, biases, wrong_masks)  # 使用错误 RNG 状态执行重计算 backward。
wrong_error = max(float(np.max(np.abs(left - right))) for left, right in zip(full_gradients, wrong_gradients))  # 计算错误重算与真实梯度的最大差异。
print("原 mask 第一层：", masks[0])  # 展示正确前向使用的 dropout mask。
print("错误重算 mask：", wrong_masks[0])  # 展示未恢复 RNG 后得到的不同 mask。
print(f"正确重算梯度误差={max_gradient_error:.3e}，错误 RNG 梯度误差={wrong_error:.3e}，额外调用={wrong_calls}")  # 量化看似能运行却训练漂移的失败。

原 mask 第一层： [1.25 1.25 1.25 1.25 1.25]
错误重算 mask： [1.25 1.25 1.25 0.   0.  ]
正确重算梯度误差=0.000e+00，错误 RNG 梯度误差=1.996e-01，额外调用=6


### 生产边界

In [6]:
policy = {"segment_layers": config["checkpoint_every"], "preserve_rng_state": True, "forbid_side_effects": ["计数器写入", "缓存追加", "外部日志重复提交"], "benchmark": ["peak_memory", "step_time", "tokens_per_second"]}  # 构造生产 activation checkpointing 配方。
print("生产 checkpoint policy：", policy)  # 展示除了开关之外必须版本化的行为合同。
print("生产替换点：真实 Transformer 还要处理 attention/MLP 分段、autocast、分布式 collective、编译图和不可重入算子。")  # 明确教学逐元素链与真实训练栈的差距。

生产 checkpoint policy： {'segment_layers': 4, 'preserve_rng_state': True, 'forbid_side_effects': ['计数器写入', '缓存追加', '外部日志重复提交'], 'benchmark': ['peak_memory', 'step_time', 'tokens_per_second']}
生产替换点：真实 Transformer 还要处理 attention/MLP 分段、autocast、分布式 collective、编译图和不可重入算子。


## 回归测试：只保护等价性和成本方向

In [7]:
assert np.allclose(full_output, checkpoint_output, atol=1e-12)  # 验证 checkpoint 前向与保存全部路径逐值一致。
assert max_gradient_error < 1e-12  # 验证恢复相同 mask 时所有权重梯度保持一致。
assert len(boundaries) < full_saved  # 验证分段方案确实减少常驻激活数量。
assert recompute_calls > 0  # 验证显存节省来自真实重计算而非漏算层。
assert wrong_error > 1e-4  # 验证错误 RNG 场景能稳定暴露梯度漂移。
print("回归测试通过：前向、梯度、显存方向、重算成本和 RNG 失败探针均有效。")  # 说明少量断言保护的核心语义。

回归测试通过：前向、梯度、显存方向、重算成本和 RNG 失败探针均有效。
